In [4]:
import os
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor

# 1. Load the file
df = pd.read_csv(r'..\..\Data Collection\nba_top_100_master.csv')

# 2. Re-establish our consensus baseline target (Average of your external tracking ranks)
df['Consensus_Target'] = -df[['Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank']].mean(axis=1)

# 3. Clean and prepare fair features matrix (Omitting Finals MVP)
classic_features = ['Career_PPG', 'Championships', 'All_NBA_PS', 'All_Defensive_Teams', 'Seasons_Played']

df_clean = df.copy()
df_clean['Career_PPG'] = df_clean['Career_PPG'].fillna(df_clean['Career_PPG'].mean())
df_clean['All_Defensive_Teams'] = df_clean['All_Defensive_Teams'].fillna(0)
df_clean['Championships'] = df_clean['Championships'].fillna(0)
df_clean['All_NBA_PS'] = df_clean['All_NBA_PS'].fillna(0)
df_clean['Seasons_Played'] = df_clean['Seasons_Played'].fillna(df_clean['Seasons_Played'].median())

# 4. Fit Gradient Boosting Decision Trees
X = df_clean[classic_features]
y = df_clean['Consensus_Target']

gbr = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
gbr.fit(X, y)

# 5. Extract Model predictions and map to a 0-100 index scale
df_clean['GBR_Raw_Prediction'] = gbr.predict(X)
min_val = df_clean['GBR_Raw_Prediction'].min()
max_val = df_clean['GBR_Raw_Prediction'].max()
df_clean['Classic_Greatness_Index'] = ((df_clean['GBR_Raw_Prediction'] - min_val) / (max_val - min_val)) * 100

# 6. Apply strictly ordered indexing ranks
df_clean['Classic_ML_Rank'] = df_clean['Classic_Greatness_Index'].rank(ascending=False, method='min').astype(int)
df_final = df_clean.sort_values('Classic_ML_Rank')

# 7. Select, format, and save variables to the new designated file
output_cols = [
    'Player_Name', 'Era_Label', 'Classic_ML_Rank', 'Classic_Greatness_Index',
    'All_NBA_Teams', 'Championships', 'Career_PPG', 'All_Defensive_Teams', 'Seasons_Played'
]
output_file_name = 'Gradient_Boosted_Metrics_Values.csv'
df_final[output_cols].to_csv(output_file_name, index=False)

In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Load files
gb_file = 'Gradient_Boosted_Metrics_Values.csv'
df_gb = pd.read_csv(gb_file)

df_master = pd.read_csv(r'..\..\Data Collection\nba_top_100_master.csv')

# 2. Get ground truth consensus target
df_master['Consensus_Rank'] = df_master[['Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank']].mean(axis=1)

# 3. Align data
df_eval = pd.merge(df_gb[['Player_Name', 'Classic_ML_Rank']], df_master[['Player_Name', 'Consensus_Rank']], on='Player_Name')
df_eval['Rank_Delta'] = df_eval['Classic_ML_Rank'] - df_eval['Consensus_Rank']

# 4. Calculate stats
mae = mean_absolute_error(df_eval['Consensus_Rank'], df_eval['Classic_ML_Rank'])
rmse = np.sqrt(mean_squared_error(df_eval['Consensus_Rank'], df_eval['Classic_ML_Rank']))
r2 = r2_score(df_eval['Consensus_Rank'], df_eval['Classic_ML_Rank'])

print("🔬 ================= GRADIENT BOOSTING DIAGNOSTICS ================= 🔬")
print(f"📈 Model R² Score (Variance Explained): {r2:.4f}")
print(f"🎯 Mean Absolute Error (Average Deviation): {mae:.2f} positions")
print(f"📉 Root Mean Squared Error (Outlier Penalty): {rmse:.2f} positions")
print("-" * 68)

# Show top 3 closest and top 3 furthest
print("\n🎯 TOP 3 PERFECT HITS / CLOSEST PREDICTIONS:")
print(df_eval.loc[df_eval['Rank_Delta'].abs().nsmallest(3).index][['Player_Name', 'Consensus_Rank', 'Classic_ML_Rank', 'Rank_Delta']].to_string(index=False))

print("\n⚠️ TOP 3 HIGHEST VARIANCE OUTLIERS:")
print(df_eval.loc[df_eval['Rank_Delta'].abs().nlargest(3).index][['Player_Name', 'Consensus_Rank', 'Classic_ML_Rank', 'Rank_Delta']].to_string(index=False))
print("=====================================================================")

🔬 ================= GRADIENT BOOSTING DIAGNOSTICS ================= 🔬
📈 Model R² Score (Variance Explained): 0.9558
🎯 Mean Absolute Error (Average Deviation): 4.78 positions
📉 Root Mean Squared Error (Outlier Penalty): 6.47 positions
--------------------------------------------------------------------

🎯 TOP 3 PERFECT HITS / CLOSEST PREDICTIONS:
        Player_Name  Consensus_Rank  Classic_ML_Rank  Rank_Delta
     Michael Jordan             1.0                1         0.0
       LeBron James             2.0                2         0.0
Kareem Abdul-Jabbar             3.0                3         0.0

⚠️ TOP 3 HIGHEST VARIANCE OUTLIERS:
     Player_Name  Consensus_Rank  Classic_ML_Rank  Rank_Delta
     Earl Monroe       85.666667              103   17.333333
     Rudy Gobert       88.666667              105   16.333333
Dave DeBusschere       97.666667              114   16.333333


In [7]:
import os
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor

# 1. Load and prepare master dataset
df_master = pd.read_csv(r'..\..\Data Collection\nba_top_100_master.csv')

# Calculate real-world tracking baseline
df_master['Consensus_Rank'] = df_master[['Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank']].mean(axis=1)

# Clean missing parameters safely
classic_features = ['Career_PPG', 'Championships', 'All_NBA_PS', 'All_Defensive_Teams', 'Seasons_Played']
df_gb_clean = df_master.copy()
df_gb_clean['Career_PPG'] = df_gb_clean['Career_PPG'].fillna(df_gb_clean['Career_PPG'].mean())
df_gb_clean['All_Defensive_Teams'] = df_gb_clean['All_Defensive_Teams'].fillna(0)
df_gb_clean['Championships'] = df_gb_clean['Championships'].fillna(0)
df_gb_clean['All_NBA_Teams'] = df_gb_clean['All_NBA_Teams'].fillna(0)
df_gb_clean['Seasons_Played'] = df_gb_clean['Seasons_Played'].fillna(df_gb_clean['Seasons_Played'].median())

# 2. Train Gradient Boosting Regressor
X_gb = df_gb_clean[classic_features]
y_gb = -df_gb_clean['Consensus_Rank'] # Inverted target baseline

gbr = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
gbr.fit(X_gb, y_gb)

# 3. Process Index Scores & Ranks
df_gb_clean['GBR_Raw_Prediction'] = gbr.predict(X_gb)
min_gb = df_gb_clean['GBR_Raw_Prediction'].min()
max_gb = df_gb_clean['GBR_Raw_Prediction'].max()
df_gb_clean['Classic_Greatness_Index'] = ((df_gb_clean['GBR_Raw_Prediction'] - min_gb) / (max_gb - min_gb)) * 100
df_gb_clean['Classic_ML_Rank'] = df_gb_clean['Classic_Greatness_Index'].rank(ascending=False, method='min').astype(int)

# 4. Filter out only the Top 20 for display
gb_top20 = df_gb_clean.sort_values(by='Classic_ML_Rank').head(20)[
    ['Classic_ML_Rank', 'Player_Name', 'Classic_Greatness_Index', 'Consensus_Rank']
]

# 5. Print Output Dashboard
print("\n🔥 ==================== GRADIENT BOOSTING MODEL: TOP 20 ==================== 🔥")
print(f"{'GB Rank':<9} | {'Player Name':<22} | {'Model Index':<11} | {'Actual Consensus Rank':<21}")
print("-" * 75)
for _, row in gb_top20.iterrows():
    print(f"{int(row['Classic_ML_Rank']):<9} | {row['Player_Name']:<22} | {row['Classic_Greatness_Index']:<11.2f} | {row['Consensus_Rank']:<21.2f}")


🔥 ==================== GRADIENT BOOSTING MODEL: TOP 20 ==================== 🔥
GB Rank   | Player Name            | Model Index | Actual Consensus Rank
---------------------------------------------------------------------------
1         | Michael Jordan         | 100.00      | 1.00                 
2         | LeBron James           | 99.49       | 2.00                 
3         | Kareem Abdul-Jabbar    | 96.91       | 3.00                 
4         | Tim Duncan             | 96.88       | 6.33                 
5         | Bill Russell           | 95.93       | 4.67                 
6         | Kobe Bryant            | 95.56       | 9.33                 
7         | Larry Bird             | 94.19       | 9.00                 
8         | Magic Johnson          | 94.18       | 5.00                 
9         | Shaquille O'Neal       | 93.35       | 7.00                 
10        | Wilt Chamberlain       | 92.98       | 8.00                 
11        | Kevin Durant           | 90.75